<hr style="border:1px solid #FEC60D">
<center><h1 style="color:#173F8A;">
Magíster en Inteligencia Artificial (MIA) <br>
Instituto de Ingeniería Matemática y Computacional (IMC)<br>
Pontificia Universidad Católica de Chile
</h1></center>
<hr style="border:1px solid #FEC60D">

<h3 style="color:#03122E;text-align:left;">INF3862-1 - Plataformas Big Data para Data Science</h3>

<center><h3 style="color:#0176DE;">
Proyecto del curso, Entrega 2<br>Hábitos de actividad y alimentación asociados a mejores indicadores de salud
</h3></center>

<center><h4 style="color:#707070;">
<b>Integrantes:</b><br>
Miguel Angel Cervantes Buleje<br>
Roger Ruben Reyes Miranda<br>
</h4></center>

<center><span style="color:#707070;">Noviembre 2026</span></center>

## Problema, entorno y datos

**Dashboard de comparación del equipo de nutrición** (E1, sección 4.1). Agrupa a los usuarios por cuartil de frecuencia cardíaca en reposo y compara, por día:
- **Actividad:** pasos y minutos de sesión.
- **Alimentación:** sodio, grasa saturada y fibra de las recetas que preparan.

**Diseño:**
- **Pipeline:** DuckDB, con las capas bronze, silver y gold.
- **Código:** `pipeline.py` (tareas del DAG) y `consultas.py` (consultas 4, 5 y 6 de la E1). Carpeta de importación: celda de entorno.

**Datos:**
- **Alcance:**
  - 61 días de mediciones;
  - 300 usuarios con reloj;
  - 522517 recetas de Food.com.
- **Origen:** `datos_proyecto_INF3862.zip`, carpeta `datos/` (detalle en `datos/README.md`).
- **Capas:**
  - bronze: `datos/datos_proyecto_INF3862/`, sin cambios;
  - silver: `datos/silver/`;
  - gold: `datos/gold/`.

In [1]:
import platform
import sys
from pathlib import Path

import duckdb
import pandas as pd

# raíz del repositorio: la carpeta que tiene pyproject.toml
RAIZ = next(c for c in (Path.cwd(), *Path.cwd().parents) if (c / "pyproject.toml").exists())
# los módulos de la carpeta del notebook si están (copia fija de una entrega); si no, los de src/
CODIGO = Path.cwd() if (Path.cwd() / "pipeline.py").exists() else RAIZ / "src"
sys.path.insert(0, str(CODIGO))

from consultas import CONSULTAS
from pipeline import USUARIOS_FOODCOM, Pipeline, cpus_visibles, medir, megabytes

meminfo = Path("/proc/meminfo")
memoria_gb = int(meminfo.read_text().split()[1]) / 1e6 if meminfo.exists() else float("nan")
print("Python", platform.python_version(), "| DuckDB", duckdb.__version__, "| pandas", pd.__version__)
print("CPU visibles:", cpus_visibles(), f"| memoria: {memoria_gb:.1f} GB | sistema:", platform.system())
print("Código desde:", CODIGO.relative_to(RAIZ))

Python 3.12.12 | DuckDB 2.0.0 | pandas 3.1.1
CPU visibles: 12 | memoria: 10.2 GB | sistema: Linux
Código desde: entregas/E2/src


## 5.2 Implementación mínima

**Tareas implementadas del DAG** (sección 5.1 del informe), cada una ejecutada con `p.tarea`, que guarda su tiempo:
- `cargar_bronze`: paso 1;
- `construir_silver`: pasos 2 y 3;
- `construir_gold`: paso 4.

Paso 5: consultas sobre gold.

### Paso 1. Cargar los datos crudos tal como vienen

In [2]:
p = Pipeline(RAIZ)
p.tarea("cargar_bronze", p.cargar_bronze)
bronze = pd.DataFrame(p.resumen_bronze(), columns=["fuente", "formato", "archivos", "filas", "MB"]).round(2)
bronze

,fuente,formato,archivos,filas,MB
0,mediciones,JSONL.GZ,1449,15540132,115.50
1,dispositivos,CSV,1,315,0.02
2,sesiones,JSONL,1,4840,0.78
3,comidas,CSV,1,22106,1.01
4,recetas,Parquet,1,522517,178.72


**¿Cómo se cargan los datos crudos tal como vienen?** DuckDB lee los archivos de `datos/datos_proyecto_INF3862/` sin modificarlos ni moverlos y unifica los esquemas de los dos modelos de reloj sin descartar filas.

- **Archivos:** 1449 de mediciones en JSON comprimido, 2 CSV (dispositivos y comidas), 1 JSONL de sesiones y 1 Parquet de recetas.
- **Esquemas de reloj:** `device_id` o `dev`; `ts_medicion` en ISO 8601 o `timestamp` en milisegundos; `ts_recepcion` o `received_at`; `frecuencia_cardiaca` o `fc`; `pasos` o `steps`; `calorias` o `kcal`.
- **Resultado:** las 15540132 mediciones llegan completas a las reglas del paso 2.

### Paso 2. Reglas de calidad

In [3]:
p.tarea("construir_silver", p.construir_silver)
calidad = pd.DataFrame(p.calidad.items(), columns=["regla o conteo", "registros"])
print("Atraso máximo entre medición y recepción:", p.atraso_maximo_dias, "días")
calidad

Atraso máximo entre medición y recepción: 4 días


,regla o conteo,registros
0,mediciones leídas,15540132
1,tipos: sin marca de tiempo o reloj,0
2,claves: reloj fuera de dispositivos.csv,0
3,duplicados: mismo reloj y marca de tiempo,345599
4,duplicados con valores distintos,0
5,claves: medición fuera del vínculo del reloj,0
6,nulos: frecuencia cardíaca nula,349931
7,rangos: frecuencia cardíaca fuera de 20-250,0
8,rangos: pasos o calorías negativos,0
9,atrasos: recibidas más de 1 hora después,8957451


**¿Qué reglas de calidad se aplican y cuántos registros afecta cada una?** Seis reglas; con casos: duplicados, nulos, claves y atrasos; sin casos: tipos y rangos (SQL de cada regla en `construir_silver` de `pipeline.py`).

- **Tipos:** marcas de tiempo de los dos relojes (ISO 8601 y milisegundos) convertidas a un solo tipo; 0 mediciones sin reloj o con una marca que no se puede convertir.
- **Claves que no calzan:**
  - mediciones de relojes que no están en `dispositivos.csv`: 0;
  - mediciones fuera del período en que su reloj estaba vinculado: 0;
  - comidas con una receta que no está en el catálogo: 48, fuera de silver, porque sin receta no tienen nutrientes.
- **Duplicados:**
  - mediciones: 345599 con reloj y marca de tiempo repetidos (2.2% de 15540132), 0 con valores distintos; son lotes reenviados y queda la primera recepción;
  - sesiones: 48 con `session_id` repetido; sin esta regla, sus minutos se sumarían dos veces;
  - comidas: ninguna repetida.
- **Nulos:** 349931 mediciones sin frecuencia cardíaca (2.3% de las 15194533 sin repetidos). Se conservan por sus pasos y calorías, fuera del promedio de reposo. En la E1 se contaron 357927, antes de quitar los repetidos: los 7996 de diferencia estaban en mediciones repetidas.
- **Rangos válidos:** 0 frecuencias fuera de 20-250 latidos por minuto; 0 pasos o calorías negativos.
- **Atrasos:**
  - más de una hora después de medirse: 8957451 mediciones (59.0% de las 15194533 sin repetidos);
  - otro día UTC: 2185094, con hasta 4 días de atraso;
  - sin descartarlas: cada una va al día en que se midió (paso 3).

### Paso 3. Capa limpia en Parquet, particionada según la E1

In [4]:
particiones = sorted(d.name for d in (p.silver / "mediciones").iterdir())
print("Particiones de mediciones:", len(particiones), "| primera:", particiones[0], "| última:", particiones[-1])
print("Mediciones en silver:", p.valor(f"select count(*) from {p.tabla(p.silver, 'mediciones')}")[0],
      f"| MB: {megabytes(p.silver / 'mediciones'):.1f} | MB en bronze: {megabytes(p.bronze / 'wearable' / 'mediciones'):.1f}")
print("Sesiones y comidas en silver:", p.valor(f"select count(*) from {p.tabla(p.silver, 'sesiones')}")[0], "|",
      p.valor(f"select count(*) from {p.tabla(p.silver, 'comidas')}")[0])

Particiones de mediciones: 61 | primera: fecha_medicion=2007-12-16 | última: fecha_medicion=2008-02-14


Mediciones en silver: 15194533 | MB: 52.2 | MB en bronze: 115.5
Sesiones y comidas en silver: 4792 | 22058


**¿Cómo se guarda la capa limpia?** Parquet con compresión ZSTD, particionado por fecha de medición como se planteó en la E1.

- **Mediciones:** las 15194533 sin repetidos, en 61 carpetas, del 2007-12-16 al 2008-02-14; 52.2 MB, menos de la mitad de los 115.5 MB de JSON comprimido de bronze.
- **Atrasadas:** en la carpeta del día UTC en que se midieron; recalcular un día local lee a lo más dos carpetas, porque la zona horaria de cada usuario desplaza su día local a lo más un día UTC.
- **Sesiones y comidas:** 4792 y 22058, sin partición, porque ocupan cerca de 1 MB cada una en bronze (0.78 y 1.01 MB, Paso 1).

### Paso 4. Tabla de hechos y dimensiones de la E1

In [5]:
p.tarea("construir_gold", p.construir_gold)
p.registrar_vistas_gold()
TABLAS_GOLD = ["hecho_usuario_dia", "hecho_usuario_reloj_dia", "hecho_comida",
               "dim_fecha", "dim_usuario", "dim_reloj", "dim_receta"]
gold = pd.DataFrame([(t, p.valor(f"select count(*) from {t}")[0],
                      len([d for d in (p.gold / t).iterdir() if d.is_dir()]))
                     for t in TABLAS_GOLD], columns=["tabla", "filas", "particiones por mes"])
# días sin mediciones: antes de vincular el primer reloj o con un reloj vinculado que no envió datos
print(p.sql("""
    with vinculo as (select usuario_key, min(cast(desde as date)) as desde from dim_reloj group by 1)
    select count(*) filter (where h.pasos is null) as dias_sin_mediciones,
           count(*) filter (where h.pasos is null and h.fecha_key < v.desde) as antes_de_vincular,
           count(*) filter (where h.fc_reposo is not null) as dias_con_fc_reposo,
           count(distinct h.usuario_key) filter (where h.fc_reposo is not null) as usuarios_con_fc_reposo
    from hecho_usuario_dia h join vinculo v using (usuario_key)""").df().to_string(index=False))
gold

 dias_sin_mediciones  antes_de_vincular  dias_con_fc_reposo  usuarios_con_fc_reposo
                1167                674               11823                     216


,tabla,filas,particiones por mes
0,hecho_usuario_dia,17497,3
1,hecho_usuario_reloj_dia,16330,3
2,hecho_comida,22058,3
3,dim_fecha,61,0
4,dim_usuario,300,0
5,dim_reloj,315,0
6,dim_receta,20654,0


**¿Cómo se construyen la tabla de hechos y las dimensiones de la E1?** Los tres hechos y las cuatro dimensiones del anexo B de la E1, en Parquet, con sus nombres y claves (`usuario_key`, `fecha_key`, `reloj_key`, `receta_key`). Hechos particionados por mes como en la E1 (3 carpetas: diciembre, enero y febrero); dimensiones sin partición.

- **`hecho_usuario_dia`:** 17497 filas, una por usuario y día local con mediciones o con comidas.
  - días con comidas y sin mediciones: 1167 (674 antes de la primera vinculación de un reloj y 493 después);
  - pasos de esos días: nulos y no cero, porque el reloj no midió.
- **Frecuencia cardíaca en reposo:** 11823 días de 216 usuarios, los mismos 216 con mediciones de madrugada de la E1. Solo horas con al menos media hora de mediciones: 30 en Pulso S, que mide cada minuto, y 6 en FitBand 2, que mide cada cinco (`wearable/LEEME.md`).
- **`hecho_usuario_reloj_dia`:** 16330 filas, una por usuario, día y reloj; la consulta 5 las agrupa por modelo.
- **`hecho_comida`:** 22058 comidas con receta en el catálogo y sus nutrientes consumidos (nutrientes por porción por las porciones de la comida).
- **Dimensiones:** 61 fechas, 300 usuarios, 315 relojes y las 20654 recetas que aparecen en las comidas.

### Paso 5. Consultas analíticas de la E1

In [6]:
resultados = {nombre: p.sql(consulta).df() for nombre, consulta in CONSULTAS.items()}
resultados["consulta_4"]

,cuartil,usuarios,fc_reposo,pasos,minutos_sesion,sodio,grasa_saturada,fibra
0,1,54,51.4,8515.0,14.6,4251.0,49.6,17.4
1,2,54,58.4,8070.0,13.4,4040.0,45.0,17.4
2,3,54,63.0,8253.0,14.6,3890.0,47.3,15.9
3,4,54,69.3,8822.0,16.0,4588.0,53.6,18.1


**¿Qué hábitos tiene cada cuartil de frecuencia cardíaca en reposo?** Ningún hábito sigue el orden de los cuartiles.

- **Cuartiles:** 54 de los 216 usuarios cada uno, con promedios de 51.4 a 69.3 latidos por minuto.
- **Actividad:**
  - pasos diarios: 8515 en el cuartil 1 (frecuencia más baja) y 8822 en el cuartil 4; el mínimo es el cuartil 2 (8070);
  - minutos de sesión: de 13.4 a 16.0.
- **Alimentación:** el cuartil 4 consume más sodio (4588 mg), más grasa saturada (53.6 g) y también más fibra (18.1 g) que los otros tres.
- **Interpretación:** estos promedios no separan al cuartil 1 (frecuencia más baja) de los demás. Es una descripción de datos sintéticos, no una relación causal.

In [7]:
# diferencia entre modelos en todo el período, la misma escala que la consulta 4
periodo = p.sql("""SELECT r.modelo, AVG(h.pasos) AS pasos FROM hecho_usuario_reloj_dia h
    JOIN dim_reloj r ON r.reloj_key = h.reloj_key GROUP BY r.modelo""").df().set_index("modelo")["pasos"]
print("Pasos diarios en el período, FitBand 2 menos Pulso S:", round(periodo["FitBand 2"] - periodo["Pulso S"], 1))
resultados["consulta_5"].pivot(index="semana", columns="modelo", values=["pasos_promedio", "filas"])

Pasos diarios en el período, FitBand 2 menos Pulso S: 77.4


pasos_promedio             filas        
modelo        FitBand 2 Pulso S FitBand 2 Pulso S
semana                                           
2007-W50         8367.0  8055.0      80.0   149.0
2007-W51         8445.0  8503.0     595.0  1066.0
2007-W52         8464.0  8232.0     636.0  1113.0
2008-W01         8424.0  8246.0     706.0  1144.0
2008-W02         8420.0  8258.0     718.0  1222.0
2008-W03         8390.0  8367.0     732.0  1267.0
2008-W04         8698.0  8378.0     751.0  1272.0
2008-W05         8315.0  8588.0     744.0  1263.0
2008-W06         8277.0  8422.0     751.0  1259.0
2008-W07         8561.0  8103.0     314.0   548.0

**¿Reportan pasos parecidos los dos modelos de reloj?** Sí, en promedio: en el período, FitBand 2 registra 77.4 pasos diarios más que Pulso S.

- **Semanas:** la diferencia es negativa en 3 y positiva en 7, de -273 pasos (2008-W05) a +458 (2008-W07).
- **Semanas incompletas:** 2007-W50 y 2008-W07, con 80 y 314 filas de FitBand 2, frente a 595-751 en las otras 8; pesan menos en el promedio del período.
- **Frente a la consulta 4:** aun si un cuartil tuviera solo usuarios de un modelo y otro solo del otro, ese reparto movería su diferencia unos 77 pasos, menos que cualquier diferencia entre cuartiles en el período (de 183 a 752 pasos).
- **Interpretación:** la consulta 4 puede juntar los dos modelos sin corregir sus pasos.

In [8]:
# comidas que cuenta la consulta 6: las de usuarios con cuartil
comidas_c6 = p.valor("""select count(*) from hecho_comida where usuario_key in
                       (select usuario_key from hecho_usuario_dia where fc_reposo is not null)""")[0]
print("Comidas de usuarios con cuartil:", comidas_c6, "| máximo de veces de una receta en un cuartil:",
      resultados["consulta_6"]["veces"].max(),
      "| mínimo en el top 10:", resultados["consulta_6"]["veces"].min())
# nombres compartidos por más de una receta: la consulta 6 agrupa también por receta_key
nombres_repetidos = p.valor("select count(*) from (select nombre from dim_receta group by 1 having count(*) > 1)")[0]
print("Nombres de receta que usan dos o más recetas:", nombres_repetidos)
resultados["consulta_6"].groupby("cuartil").head(3)

Comidas de usuarios con cuartil: 15889 | máximo de veces de una receta en un cuartil: 4 | mínimo en el top 10: 2
Nombres de receta que usan dos o más recetas: 469


,cuartil,nombre,veces
0,1,Bourbon Chicken,4
1,1,Spinach Artichoke Dip,3
2,1,Bacon Potato Frittata,2
10,2,Chocolate Cream Cheese Pound Cake,3
11,2,5 Can Ww Soup,2
12,2,Absolute Best Ever Lasagna,2
20,3,Authentic Italian Meatballs,3
21,3,Simple Whole Wheat Muffins,3
22,3,2bleu's Sweet Chicken and Gravy,2
30,4,Applesauce Oatmeal,3


**¿Qué recetas prepara más cada cuartil?** Ninguna se repite lo suficiente para distinguir a los cuartiles.

- **Más repetida:** Bourbon Chicken, 4 veces en el cuartil 1, entre 15889 comidas de usuarios con cuartil (el mismo conteo del anexo A de la E1).
- **Agrupación:** también por `receta_key`, porque 469 nombres corresponden a dos o más recetas distintas.
- **Top 10:** el de cada cuartil termina en empates de 2 veces, ordenados por nombre.

### Reintentos e idempotencia (5.1)

In [9]:
# repetir las tareas silver y gold debe dejar las mismas filas, sin duplicar
antes = {t: p.valor(f"select count(*) from {t}")[0] for t in TABLAS_GOLD}
p.construir_silver()
p.construir_gold()
p.registrar_vistas_gold()
despues = {t: p.valor(f"select count(*) from {t}")[0] for t in TABLAS_GOLD}
print("Mismas filas después de repetir:", antes == despues, "|", despues["hecho_usuario_dia"], "filas en hecho_usuario_dia")

Mismas filas después de repetir: True | 17497 filas en hecho_usuario_dia


**¿Cómo se evita duplicar registros si una carga se repite?** Cada tarea borra y vuelve a escribir completa la carpeta que produce, y silver quita los repetidos por reloj y marca de tiempo antes de escribir.

- **Repetición:** esta celda vuelve a ejecutar silver y gold después de una corrida completa; las siete tablas quedan con las mismas filas (17497 en `hecho_usuario_dia`).
- **Falla a la mitad:** deja una carpeta incompleta, que el reintento borra entera.

## 5.3 Medir y explicar

In [10]:
# consulta 4 sobre hecho_usuario_dia guardado en CSV y en Parquet
csv_hecho = p.gold / "csv" / "hecho_usuario_dia.csv"
csv_hecho.parent.mkdir(exist_ok=True)
p.sql(f"copy (select * from hecho_usuario_dia) to '{csv_hecho}' (header)")
consulta_csv = CONSULTAS["consulta_4"].replace("FROM hecho_usuario_dia", f"FROM read_csv_auto('{csv_hecho}')")
consulta_parquet = CONSULTAS["consulta_4"]
mismo_resultado = p.sql(consulta_csv).df().equals(p.sql(consulta_parquet).df())

formatos = pd.DataFrame({
    "formato": ["CSV", "Parquet"],
    "MB en disco": [megabytes(csv_hecho.parent), megabytes(p.gold / "hecho_usuario_dia")],
    "ms (mediana de 7)": [medir(lambda: p.sql(consulta_csv).fetchall()),
                          medir(lambda: p.sql(consulta_parquet).fetchall())],
}).round(2)
print("Mismo resultado en CSV y Parquet:", mismo_resultado)
formatos

Mismo resultado en CSV y Parquet: True


,formato,MB en disco,ms (mediana de 7)
0,CSV,2.15,266.94
1,Parquet,0.52,24.02


In [11]:
# las columnas que lee la consulta 4 frente a las que tiene el hecho
columnas_hecho = p.sql("select * from hecho_usuario_dia limit 0").columns
columnas_consulta = ["usuario_key", "fecha_key", "fc_reposo", "pasos", "minutos_sesion", "sodio", "grasa_saturada", "fibra"]
print("Columnas del hecho:", len(columnas_hecho), "| que lee la consulta 4:", len(columnas_consulta))
print("Filas del hecho:", p.valor("select count(*) from hecho_usuario_dia")[0],
      "| archivos Parquet:", len(list((p.gold / "hecho_usuario_dia").rglob("*.parquet"))))

Columnas del hecho: 18 | que lee la consulta 4: 8
Filas del hecho: 17497 | archivos Parquet: 3


**¿Cuánto tarda la consulta 4 sobre CSV y sobre Parquet, y cuánto ocupa cada uno?** Parquet: 24.02 ms y 0.52 MB; CSV: 266.94 ms y 2.15 MB; las dos dan el mismo resultado.

- **Medición:** mediana de 7 ejecuciones en el contenedor de la sección de entorno.
- **Columnas:** la consulta lee 8 de las 18 columnas de `hecho_usuario_dia`.
- **Archivos:** el hecho en Parquet son 3 archivos, uno por mes; la consulta pide el período completo y lee los 3.

## 5.4 ¿Realmente necesitamos Big Data?

In [12]:
DIAS_ANIO = 365                   # días de un año
usuarios_reloj = p.valor("select count(distinct AuthorId) from dispositivos")[0]
usuarios_foodcom = p.valor(f"select count(distinct AuthorId) from read_parquet('{p.bronze / USUARIOS_FOODCOM}')")[0]
dias = p.dias_recepcion
filas_crudas = p.calidad["mediciones leídas"]
filas_usuario_dia = filas_crudas / usuarios_reloj / dias
mb_bronze_fila = megabytes(p.bronze / "wearable" / "mediciones") / filas_crudas
mb_silver_fila = megabytes(p.silver / "mediciones") / p.calidad["mediciones en silver"]
segundos_fila = (p.tiempos["cargar_bronze"] + p.tiempos["construir_silver"]) / filas_crudas
print(f"Usuarios con reloj: {usuarios_reloj} | usuarios de Food.com: {usuarios_foodcom} | días: {dias}")
print(f"Mediciones por usuario y día: {filas_usuario_dia:.1f} | bytes por medición: bronze {mb_bronze_fila * 1e6:.2f}, silver {mb_silver_fila * 1e6:.2f}")
print(f"Tiempo de bronze y silver: {p.tiempos['cargar_bronze'] + p.tiempos['construir_silver']:.1f} s "
      f"| memoria máxima del proceso: {p.memoria_maxima_gb():.1f} GB de {memoria_gb:.1f} GB")

escenarios = {"hoy (300 usuarios, 61 días)": (usuarios_reloj, dias),
              "Food.com, 1 año": (usuarios_foodcom, DIAS_ANIO),
              "Food.com, 3 años": (usuarios_foodcom, 3 * DIAS_ANIO),
              "Food.com x100, 3 años": (100 * usuarios_foodcom, 3 * DIAS_ANIO)}
escala = pd.DataFrame([{
    "escenario": nombre,
    "mediciones por día (millones)": usuarios * filas_usuario_dia / 1e6,
    "mediciones (miles de millones)": usuarios * filas_usuario_dia * d / 1e9,
    "bronze (GB)": usuarios * filas_usuario_dia * d * mb_bronze_fila / 1e3,
    "silver (GB)": usuarios * filas_usuario_dia * d * mb_silver_fila / 1e3,
    "lote diario (min)": usuarios * filas_usuario_dia * segundos_fila / 60,
    "reproceso completo (h)": usuarios * filas_usuario_dia * d * segundos_fila / 3600,
} for nombre, (usuarios, d) in escenarios.items()])
escala.round(2)

Usuarios con reloj: 300 | usuarios de Food.com: 271907 | días: 61
Mediciones por usuario y día: 849.2 | bytes por medición: bronze 7.43, silver 3.43
Tiempo de bronze y silver: 20.7 s | memoria máxima del proceso: 6.0 GB de 10.2 GB


,escenario,mediciones por día (millones),mediciones (miles de millones),bronze (GB),silver (GB),lote diario (min),reproceso completo (h)
0,"hoy (300 usuarios, 61 días)",0.25,0.02,0.12,0.05,0.01,0.01
1,"Food.com, 1 año",230.90,84.28,626.39,289.20,5.13,31.18
2,"Food.com, 3 años",230.90,252.84,1879.17,867.59,5.13,93.55
3,"Food.com x100, 3 años",23090.00,25283.55,187916.50,86758.75,512.61,9355.16


**¿Qué volumen tendría la app con todos los usuarios de Food.com, y cuánto tardaría en una máquina?** Estimaciones, no mediciones: extrapolación lineal desde esta corrida (bytes por medición y segundos de bronze y silver por medición; 20.7 s en bronze y silver).

- **Supuesto:** los 271907 usuarios que reseñan en Food.com tienen reloj y generan, como los 300 actuales, 849.2 mediciones por día.
- **1 año:** 84.28 mil millones de mediciones; 626.39 GB en bronze y 289.20 GB en silver; lote diario de 230.9 millones de mediciones en unos 5.13 minutos.
- **3 años:** 252.84 mil millones de mediciones; 1879.17 GB en bronze y 867.59 GB en silver; el mismo lote diario; volver a procesar los 3 años, unas 93.6 horas.
- **x100 a 3 años:** lote diario en 512.61 minutos (8.5 horas); bronze de 187916.5 GB.
- **Memoria:** el proceso del notebook llegó a 6.0 GB de los 10.2 GB del contenedor, el pico de todas sus tareas, con 15540132 mediciones.